In [19]:
from sklearn.datasets import make_classification
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch import nn

In [20]:
x,y=make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=10,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

In [21]:
x=torch.tensor(x,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.float32)

In [22]:
class CustomDataset(Dataset):
    def __init__(self,x,y):
        self.x=x
        self.y=y
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

In [23]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [24]:
dataset=CustomDataset(x_train,y_train)

In [25]:
DataLoader=DataLoader(dataset,batch_size=50,shuffle=True)

In [29]:
class mysimplemodel(nn.Module):
    def __init__(self, num_features):
      super().__init__()
      self.network = nn.Sequential(
        nn.Linear(num_features, 10),
        nn.ReLU(),
        nn.Linear(10, 1),
        nn.Sigmoid()
    )
    def forward(self, features):

        out = self.network(features)

        return out

In [33]:
learning_rate = 0.1
epochs = 10000
loss_function = nn.BCELoss()


In [34]:
model = mysimplemodel(x_train.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loop
for epoch in range(epochs):
    for batch_idx, (features, labels) in enumerate(DataLoader):
      # forward pass
      y_pred = model(features)

      # loss calculate
      loss = loss_function(y_pred, labels.view(-1,1))

      # backward pass
      loss.backward()

      # parameters update
      optimizer.step()

      # clear gradients
      optimizer.zero_grad()

    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Streaming output truncated to the last 5000 lines.
Epoch: 5001, Loss: 4.330537194618955e-05
Epoch: 5002, Loss: 0.0002893941127695143
Epoch: 5003, Loss: 0.0004629400500562042
Epoch: 5004, Loss: 0.0007704318850301206
Epoch: 5005, Loss: 0.0007705455063842237
Epoch: 5006, Loss: 0.00033026799792423844
Epoch: 5007, Loss: 0.0008646731730550528
Epoch: 5008, Loss: 0.0003350533661432564
Epoch: 5009, Loss: 0.0003011486551258713
Epoch: 5010, Loss: 0.00025541166542097926
Epoch: 5011, Loss: 0.0002435657224850729
Epoch: 5012, Loss: 0.00048821597010828555
Epoch: 5013, Loss: 0.0004372894763946533
Epoch: 5014, Loss: 0.00041254141251556575
Epoch: 5015, Loss: 0.0007295419345609844
Epoch: 5016, Loss: 0.0006298270891420543
Epoch: 5017, Loss: 0.0005073031643405557
Epoch: 5018, Loss: 0.0010343915782868862
Epoch: 5019, Loss: 0.00039246754022315145
Epoch: 5020, Loss: 0.0005977626424282789
Epoch: 5021, Loss: 0.0005481251864694059
Epoch: 5022, Loss: 0.0012369969626888633
Epoch: 5023, Loss: 0.0012462949380278587
E

In [53]:
with torch.no_grad():
  y_pred_proba = model.forward(x_test)
  test_loss = loss_function(y_pred_proba, y_test.view(-1,1))
  print(f'Test Loss: {test_loss.item()}')
  y_pred = (y_pred_proba > 0.5).float()
  accuracy = (y_pred == y_test.view(-1,1)).float().mean()
  print(f'Test Accuracy: {accuracy.item()}')

Test Loss: 2.324397563934326
Test Accuracy: 0.9100000262260437


In [55]:
with torch.no_grad():
    y_pred_proba = model.forward(x_train)
    y_pred = (y_pred_proba > 0.5).float()
    accuracy = (y_pred == y_train.view(-1,1)).float().mean()
    print(f'Training Accuracy: {accuracy.item()}')

Training Accuracy: 1.0
